 ## Common Time Series Analysis Techniques

In [2]:
import yfinance as yf

# Load 5days of TSLA stock pricing information
ticker = 'TSLA'
tkr = yf.Ticker(ticker)
df=tkr.history(period='5d')
df

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2025-06-05 00:00:00-04:00,322.489990,324.549988,273.209991,284.700012,287499800,0.0,0.0
2025-06-06 00:00:00-04:00,298.829987,305.500000,291.140015,295.140015,164747700,0.0,0.0
2025-06-09 00:00:00-04:00,285.959991,309.829987,281.850006,308.579987,140908900,0.0,0.0
2025-06-10 00:00:00-04:00,314.940002,327.829987,310.670013,326.089996,151256500,0.0,0.0
2025-06-11 00:00:00-04:00,334.399994,335.500000,322.500000,326.429993,121851400,0.0,0.0


## Calculating Percentage Changes

In [3]:
import pandas as pd

# Print a concatenated dataframe of the current days close information including 2 days prior close prices
print(pd.concat([df["Close"], df["Close"].shift(2)], axis=1, keys=["Close","2DaysShift"]))

                                Close  2DaysShift
Date                                             
2025-06-05 00:00:00-04:00  284.700012         NaN
2025-06-06 00:00:00-04:00  295.140015         NaN
2025-06-09 00:00:00-04:00  308.579987  284.700012
2025-06-10 00:00:00-04:00  326.089996  295.140015
2025-06-11 00:00:00-04:00  326.429993  308.579987


In [4]:
# Calulcate the percent change in price from today and 2 days ago
(df["Close"]-df["Close"].shift(2))/df["Close"].shift(2)

Date
2025-06-05 00:00:00-04:00         NaN
2025-06-06 00:00:00-04:00         NaN
2025-06-09 00:00:00-04:00    0.083878
2025-06-10 00:00:00-04:00    0.104865
2025-06-11 00:00:00-04:00    0.057846
Name: Close, dtype: float64

In [7]:
import numpy as np

# Calculate the natural logirithm percent change of todays prices vs 2 days ago price
df['2DaysRise'] = np.log(df["Close"]/df["Close"].shift(2))
print(df[["Close",'2DaysRise']])

                                Close  2DaysRise
Date                                            
2025-06-05 00:00:00-04:00  284.700012        NaN
2025-06-06 00:00:00-04:00  295.140015        NaN
2025-06-09 00:00:00-04:00  308.579987   0.080545
2025-06-10 00:00:00-04:00  326.089996   0.099724
2025-06-11 00:00:00-04:00  326.429993   0.056234


## Rolling Window Calculations

In [8]:
# Calculate the rolling 2 day avereage price 
df["2DaysAvg"] = df["Close"].shift(1).rolling(2).mean()
print(df[["Close","2DaysAvg"]])

                                Close    2DaysAvg
Date                                             
2025-06-05 00:00:00-04:00  284.700012         NaN
2025-06-06 00:00:00-04:00  295.140015         NaN
2025-06-09 00:00:00-04:00  308.579987  289.920013
2025-06-10 00:00:00-04:00  326.089996  301.860001
2025-06-11 00:00:00-04:00  326.429993  317.334991


## Calculating the Percentage Change of a Rolling Average

In [9]:
# Calculate the 2 days average rise (logarithm) in price
df['2DaysAvgRise'] = np.log(df["Close"]/df["Close"].shift(1).rolling(2).mean())
print(df[["Close","2DaysAvg","2DaysAvgRise"]])

                                Close    2DaysAvg  2DaysAvgRise
Date                                                           
2025-06-05 00:00:00-04:00  284.700012         NaN           NaN
2025-06-06 00:00:00-04:00  295.140015         NaN           NaN
2025-06-09 00:00:00-04:00  308.579987  289.920013      0.062376
2025-06-10 00:00:00-04:00  326.089996  301.860001      0.077210
2025-06-11 00:00:00-04:00  326.429993  317.334991      0.028258


## Multivariate Time Series

In [11]:
# Load the stock prices for 56 companies over a 5 day period
stocks = pd.DataFrame()
tickers = ["MSFT","TSLA","GM","AAPL","ORCL","AMZN"]
for ticker in tickers:
    tkr = yf.Ticker(ticker)
    hist = tkr.history(period='5d')
    hist = pd.DataFrame(hist[["Close"]].rename(columns={"Close":ticker}))
    if stocks.empty:
        stocks=hist
    else:
        stocks=stocks.join(hist)
stocks

,MSFT,TSLA,GM,AAPL,ORCL,AMZN
Date,,,,,,
2025-06-05 00:00:00-04:00,467.679993,284.700012,47.099998,200.630005,171.139999,207.910004
2025-06-06 00:00:00-04:00,470.380005,295.140015,47.470001,203.919998,174.020004,213.570007
2025-06-09 00:00:00-04:00,472.750000,308.579987,47.930000,201.449997,177.149994,216.979996
2025-06-10 00:00:00-04:00,470.920013,326.089996,48.930000,202.669998,177.479996,217.610001
2025-06-11 00:00:00-04:00,472.619995,326.429993,49.869999,198.779999,176.380005,213.199997


## Processing Multivariate Time Series

In [12]:
# Filter only stocks that did not have a price drop by 3% in the past day
stocks_to_keep = []
for i in stocks.columns:
    if stocks[stocks[i]/stocks[i].shift(1)<0.97].empty:
        stocks_to_keep.append(i)
print(stocks_to_keep)

['MSFT', 'TSLA', 'GM', 'AAPL', 'ORCL', 'AMZN']


## Analyzing Dependencies Between Variable

In [13]:
# Load 1 month of TSLA stock
import yfinance as yf
import numpy as np
ticker = 'TSLA'
tkr = yf.Ticker(ticker)
df = tkr.history(period='1mo')
df.head(5)

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2025-05-12 00:00:00-04:00,321.989990,322.209991,311.500000,318.380005,112826700,0.0,0.0
2025-05-13 00:00:00-04:00,320.000000,337.589996,316.799988,334.070007,136992600,0.0,0.0
2025-05-14 00:00:00-04:00,342.500000,350.000000,337.000000,347.679993,136997300,0.0,0.0
2025-05-15 00:00:00-04:00,340.339996,346.140015,334.720001,342.820007,97882600,0.0,0.0
2025-05-16 00:00:00-04:00,346.239990,351.619995,342.329987,349.980011,95895700,0.0,0.0


In [14]:
# Rename columns
df = df[["Close","Volume"]].rename(columns={"Close":"Price"})

In [15]:
# Calculate the price rise over the past day
df["priceRise"] = np.log(df["Price"]/df["Price"].shift(1))

In [18]:
# Calculate the volume rise over the past day
df["volumeRise"] = np.log(df["Volume"]/df["Volume"].shift(1))

In [20]:
# Show price rise above 1%
print(df[abs(df["priceRise"]>0.01)])

                                Price     Volume  priceRise  volumeRise
Date                                                                   
2025-05-13 00:00:00-04:00  334.070007  136992600   0.048105    0.194074
2025-05-14 00:00:00-04:00  347.679993  136997300   0.039932    0.000034
2025-05-16 00:00:00-04:00  349.980011   95895700   0.020670   -0.020508
2025-05-22 00:00:00-04:00  341.040009   97113400   0.019004   -0.052566
2025-05-27 00:00:00-04:00  362.890015  120146400   0.067097    0.350129
2025-06-06 00:00:00-04:00  295.140015  164747700   0.036014   -0.556807
2025-06-09 00:00:00-04:00  308.579987  140908900   0.044531   -0.156302
2025-06-10 00:00:00-04:00  326.089996  151256500   0.055192    0.070863


In [21]:
# Find average volume rise over the past month
print(df["volumeRise"].mean().round(4))

0.0037


In [22]:
# Find average volume rise for days with an absolute price rise greater than 0.01
print(df[abs(df["priceRise"]>0.01)]["volumeRise"].mean().round(4))

# The number below is larger than the average above, indicating a possible correlation betwen price and volume

-0.0214
